# Constant Propagation with Netlist Carpentry

Constant propagation is a pretty useful optimization technique where signals that are tied to fixed logic values (0 or 1) are substituted directly into the circuit.
This enables the simplification of logic expressions.
It also removes gates whose outputs become predetermined, reducing circuit area, power, and overall delay.
Consider the following example structure:

```mermaid
flowchart LR
    In1([Input: 1])
    In2([Input: 0])
    Gate{{AND Gate}}
    SomeWire([SomeWire: 0])
    SubModule{{Some Submodule}}
    
    In1 --> Gate
    In2 --> Gate
    Gate --> SomeWire
    SomeWire --> SubModule
```

If the inputs of the AND Gate are tied to 0 and 1 respectively, it is obvious that the output will always be 0 as well.
Accordingly, the AND Gate may be removed and the corresponding input of the downstream submodule instance can be tied to 0.
This removes unnecessary instances and reduces circuit area and power consumption.


```mermaid
flowchart LR
    SomeWire((0))
    SubModule{{Some Submodule}}

    SomeWire --> SubModule
```

Before we start, the cell below defines a small helper for the graphs in this notebook.
Tied (constant) inputs are not drawn as nodes in a circuit graph, since they are not connected to a wire.
So that it is still visible which gates have constant inputs, the helper highlights them:

- **Gold rounded rectangles** are gates with at least one constant input.
- **Red rhombuses** are inverters (`NOT` gates).

See the notebook about interactive circuit graphs for more information about formatting nodes.

In [1]:
from netlist_carpentry import Instance
from netlist_carpentry.utils.gate_lib import NotGate
from netlist_carpentry.utils.gate_lib_base_classes import PrimitiveGate
from netlist_carpentry.vis.dynamic import CytoscapeGraph
from netlist_carpentry.vis.dynamic.ipycytoscape import GraphDataDict
from netlist_carpentry.vis.styling import FormatDefinition


def has_tied_input(node_id: str, node_data: GraphDataDict) -> bool:
    obj = node_data["ndata"]
    return isinstance(obj, PrimitiveGate) and any(ps.is_tied for port in obj.ports.values() if port.is_input for _, ps in port)


def is_inverter(node_id: str, node_data: GraphDataDict) -> bool:
    return isinstance(node_data["ndata"], NotGate)


def show_constants(module) -> None:
    cyto = CytoscapeGraph(module_graph=module.graph())
    cyto.formats.add_format(".tied", FormatDefinition(background_color="#FBD100", shape="round-rectangle"))
    cyto.formats.add_format(".inverter", FormatDefinition(background_color="#FB1100", shape="rhomboid"))
    cyto.format_nodes(has_tied_input, ".tied")
    cyto.format_nodes(is_inverter, ".inverter")
    cyto.show()

Execute the cell below to create a circuit with a module, with a similar structure as in the first graph.
The AND gate is highlighted, because both of its inputs are tied.

In [2]:
from netlist_carpentry import Circuit, Module, Direction
from netlist_carpentry.utils.gate_factory import and_gate

c = Circuit(name="c")
m = c.create_module(name="m")
submodule = Module(name="Submodule")
submodule.create_port("IN", Direction.IN)

sub_inst = m.create_instance(submodule, "SubmoduleInst")
and_inst = and_gate(m, "and_inst", Y=sub_inst.ports["IN"])
and_inst.ports["A"].tie_signal(1)
and_inst.ports["B"].tie_signal(0)
show_constants(m)

Output()

Now to start the constant propagation algorithm, there are two approaches.
The easiest is to execute `Module.optimize`, which runs a bunch of optimizations, with constant propagation being one of them.
However, to only execute the constant propagation, the task can be imported directly from the corresponding module `routines.opt.constant_folds`, as shown in the cell below.


<div class="admonition info alert alert-info">
  <strong>Info:</strong> The constant propagation is executed by the function <b>opt_constant_propagation</b>.
  However, there are other functions inside the <b>constant_folds</b> module.
  To execute all constant optimization tasks, use the function <b>opt_constant</b>.
  For simplicity, the function <b>opt_constant_propagation</b> is used here!
</div>



In [3]:
# Alternative: from netlist_carpentry.routines.opt import opt_constant
from netlist_carpentry.routines.opt.constant_folds import opt_constant_propagation

print(f"Module {m.name} now has these instances: {', '.join(inst for inst in m.instances)}")
print(f"Is Port IN of Instance {sub_inst.name} tied? {sub_inst.ports["IN"].is_tied}")
print(f"Port IN of Instance {sub_inst.name} has this driver: {sub_inst.ports["IN"].driver()}")


any_constants_propagated = opt_constant_propagation(m)
print(f"Any constants propagated? {any_constants_propagated}")
print(f"Module {m.name} now has these instances: {', '.join(inst for inst in m.instances)}")

print(f"Is Port IN of Instance {sub_inst.name} tied? {sub_inst.ports["IN"].is_tied}")
print(f"Port IN of Instance {sub_inst.name} now has this value: {sub_inst.ports["IN"].signal}")

Module m now has these instances: SubmoduleInst, and_inst
Is Port IN of Instance SubmoduleInst tied? False
Port IN of Instance SubmoduleInst has this driver: {0: [PortSegment(m.and_inst.Y.0, Signal:x)]}


Any constants propagated? True
Module m now has these instances: SubmoduleInst
Is Port IN of Instance SubmoduleInst tied? True
Port IN of Instance SubmoduleInst now has this value: 0


The AND gate is gone and the submodule instance is the only one left.
Its input `IN` is now tied to the value the AND gate would have produced, which is 0.

In [4]:
show_constants(m)

Output()

## Partially Constant Inputs

A gate does not need constant values at *all* of its inputs to be removed.
Often, a single constant input is enough to simplify a gate.
Whatever the value of the other input `x` is, there are only three possible outcomes for a gate with two inputs, where one of them is constant:

1. The output is **constant**, no matter what `x` is (e.g. `x AND 0` is always 0).
2. The output is **equal to `x`** (e.g. `x AND 1` is always `x`). The gate is merely a wire and can be replaced by one.
3. The output is the **inverse of `x`** (e.g. `x XOR 1` is always `!x`). The gate can be replaced by an inverter, which is a smaller and faster gate.

The following table shows all combinations that are handled by the constant propagation:

| Gate | Constant input | Result | Action |
|------|----------------|--------|--------|
| AND  | 0 | always 0 | output is tied to 0 |
| AND  | 1 | `x` | output is connected to `x` |
| NAND | 0 | always 1 | output is tied to 1 |
| NAND | 1 | `!x` | gate is replaced by an inverter |
| OR   | 1 | always 1 | output is tied to 1 |
| OR   | 0 | `x` | output is connected to `x` |
| NOR  | 1 | always 0 | output is tied to 0 |
| NOR  | 0 | `!x` | gate is replaced by an inverter |
| XOR  | 0 | `x` | output is connected to `x` |
| XOR  | 1 | `!x` | gate is replaced by an inverter |
| XNOR | 1 | `x` | output is connected to `x` |
| XNOR | 0 | `!x` | gate is replaced by an inverter |

It makes no difference at which of the two inputs the constant is applied.

```mermaid
flowchart LR
    subgraph Before
        X1([x]) --> G1{{AND}}
        C1([1]) --> G1
        G1 --> Y1([y])
    end
    subgraph After
        X2([x]) --> Y2([y])
    end
```

```mermaid
flowchart LR
    subgraph Before
        X3([x]) --> G3{{XOR}}
        C3([1]) --> G3
        G3 --> Y3([y])
    end
    subgraph After
        X4([x]) --> N4{{NOT}} --> Y4([y])
    end
```

The cell below creates a module with one gate for each of the three outcomes.
All three gates are highlighted, since each one has a constant input.

In [5]:
from netlist_carpentry.utils.gate_factory import xor_gate

c2 = Circuit(name="c2")
m2 = c2.create_module(name="m2")
a = m2.create_port("a", Direction.IN)
b = m2.create_port("b", Direction.IN)
d = m2.create_port("d", Direction.IN)
y_const = m2.create_port("y_const", Direction.OUT)
y_wire = m2.create_port("y_wire", Direction.OUT)
y_inv = m2.create_port("y_inv", Direction.OUT)

and_zero = and_gate(m2, "and_zero", A=a, Y=y_const)
and_zero.ports["B"].tie_signal(0)
and_one = and_gate(m2, "and_one", A=b, Y=y_wire)
and_one.ports["B"].tie_signal(1)
xor_one = xor_gate(m2, "xor_one", A=d, Y=y_inv)
xor_one.ports["B"].tie_signal(1)

show_constants(m2)

Output()

Execute the cell below to run the constant propagation.
Afterwards, the three gates are gone and the outputs behave as described above.
Only the inverter, which replaced the XOR gate, is left.

In [6]:
print(f"Instances before: {', '.join(m2.instances)}")
print(f"Any constants propagated? {opt_constant_propagation(m2)}")
print(f"Instances after: {', '.join(f'{name} ({inst.instance_type})' for name, inst in m2.instances.items())}")

print(f"y_const is tied to: {m2.ports['y_const'].signal}")
print(f"y_wire now has this driver: {m2.ports['y_wire'].driver()}")
print(f"y_inv now has this driver: {m2.ports['y_inv'].driver()}")

show_constants(m2)

Instances before: and_zero, and_one, xor_one


Any constants propagated? True
Instances after: xor_one_inv0 (§not)
y_const is tied to: 0
y_wire now has this driver: {0: [PortSegment(m2.b.0, Signal:x)]}
y_inv now has this driver: {0: [PortSegment(m2.xor_one_inv0.Y.0, Signal:x)]}


Output()

<div class="admonition info alert alert-info">
  <strong>Info:</strong> The new inverter is named after the gate it replaces, with the suffix <b>_inv</b> and the bit index, e.g. <b>xor_one_inv0</b>.
  The number of gates does not change in this case (one gate is replaced by one gate), but the inverter is smaller and faster than the gate it replaced.
  Furthermore, the inverter may enable further optimizations.
</div>

## Buses

For gates that are wider than one bit, all rules are applied bit by bit.
A 4-bit `XOR` gate with `B = 4'b1010` will, for example, get an inverter for each bit where `B` is 1, and simply pass through the other bits.
A gate is only removed if **every** bit of its output can be simplified.
If just one bit has no matching rule (e.g. because both of its inputs are non-constant), the whole gate is left untouched.

## Multiplexers with a Constant Select Signal

The select input `S` of a multiplexer decides which of its data inputs is forwarded.
If `S` is constant, it is already known which input is selected, so the multiplexer is just a wire (or a constant, if the selected input is constant as well).
The other inputs are irrelevant and are ignored.
The cell below creates a multiplexer with `S` tied to 1, so that its output follows input `D1`.

In [7]:
from netlist_carpentry.utils.gate_factory import multiplexer

c3 = Circuit(name="c3")
m3 = c3.create_module(name="m3")
d0 = m3.create_port("d0", Direction.IN)
d1 = m3.create_port("d1", Direction.IN)
y_mux = m3.create_port("y", Direction.OUT)

mux = multiplexer(m3, "mux", D_ports=[d0, d1], Y=y_mux)
mux.ports["S"].tie_signal(1)

show_constants(m3)

Output()

Executing the cell below removes the multiplexer.
The output `y` is connected directly to `d1`, while `d0` is no longer used by the multiplexer.

In [8]:
print(f"Any constants propagated? {opt_constant_propagation(m3)}")
print(f"Module {m3.name} now has these instances: {', '.join(m3.instances) or '(none)'}")
print(f"Output y now has this driver: {m3.ports['y'].driver()}")

show_constants(m3)

Any constants propagated? True
Module m3 now has these instances: (none)
Output y now has this driver: {0: [PortSegment(m3.d1.0, Signal:x)]}


Output()

<div class="admonition info alert alert-info">
  <strong>Info:</strong> This is different from the function <b>opt_constant_mux_inputs</b>, which handles multiplexers whose <i>data</i> inputs are constant (e.g. <b>D0 = 0</b> and <b>D1 = 1</b>).
  In this case, the output is equal to the select signal.
  Both functions are executed by <b>opt_constant</b>.
</div>

## Chains of Gates

Removing a gate can make the next gate simplifiable.
The propagation therefore repeats until nothing changes anymore.
In the cell below, the first gate is an OR gate with a constant 1 input, so its output is always 1.
This turns the XOR gate behind it into an `XOR` with a constant 1 input, which is an inverter of `x`.

In [9]:
from netlist_carpentry.utils.gate_factory import or_gate

c4 = Circuit(name="c4")
m4 = c4.create_module(name="m4")
ctrl = m4.create_port("ctrl", Direction.IN)
x = m4.create_port("x", Direction.IN)
y_chain = m4.create_port("y", Direction.OUT)

or_inst = or_gate(m4, "or_inst", A=ctrl)
or_inst.ports["B"].tie_signal(1)
xor_inst = xor_gate(m4, "xor_inst", A=or_inst.ports["Y"], B=x, Y=y_chain)

print(f"Instances before: {', '.join(m4.instances)}")
opt_constant_propagation(m4)
print(f"Instances after: {', '.join(f'{name} ({inst.instance_type})' for name, inst in m4.instances.items())}")

Instances before: or_inst, xor_inst


Instances after: xor_inst_inv0 (§not)
